# 01. Обучение статических моделей латинского языка

    Обучаем **четыре серии по десять моделей**: ранний / поздний период × `min_count=5 / 50`.
    Архитектура — CBOW, окно 5, 100 измерений, 10 эпох; символьные n-граммы отключены.
    Это следует ранее выбранной исследовательской конфигурации по Лунарди и работам LiLa.

In [1]:
from pathlib import Path
import sys

# При необходимости укажите каталог на Google Drive:
PROJECT_ROOT = None  # например: '/content/drive/MyDrive/latinise-humanities'

candidates = [Path(PROJECT_ROOT)] if PROJECT_ROOT else [Path.cwd(), Path.cwd().parent, Path.cwd() / 'latinise-humanities']
root = next((p.resolve() for p in candidates if (p / 'src/core.py').exists()), None)
if root is None:
    raise FileNotFoundError('Укажите PROJECT_ROOT — папку репозитория.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print('Проект:', root)

Проект: C:\Users\Admin\Documents\GitHub\latinise-humanities


In [2]:
from src import core
import pandas as pd
from IPython.display import display
settings = core.config()
display(pd.DataFrame([settings]))
display(pd.read_csv(core.RESULTS / 'corpus_summary.csv'))

,min_counts,vector_size,window,epochs,negative,sample,seeds
0,"[5, 50]",100,5,10,5,0.001,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"


,period,texts,sentences,tokens,lemmas,min_count_5,min_count_50
0,pre180,387,194612,3569242,61409,18806,5548
1,post180,391,222276,3342290,67409,19056,5584


## Обучение

    Модели сохраняются в `results/runs/`. Если файл данного запуска **уже существует**, его повторное обучение пропускается. Это удобно в Google Colab.

    **Важно:** после изменения обучающих текстов удалите или переместите старые модели. Программа не сравнивает файлы по контрольным суммам.

In [3]:
log = []
for period in core.SLICES:
    for min_count in settings['min_counts']:
        for seed in settings['seeds']:
            path, status = core.train_one(period, min_count, seed)
            print(period, f'min_count={min_count}', f'seed={seed}:', status)
            log.append({'period': period, 'min_count': min_count,
                        'seed': seed, 'status': status})
core.RESULTS.mkdir(parents=True, exist_ok=True)
pd.DataFrame(log).to_csv(core.RESULTS / 'training_log.csv', index=False)

pre180 min_count=5 seed=1: обучена
pre180 min_count=5 seed=2: обучена
pre180 min_count=5 seed=3: обучена
pre180 min_count=5 seed=4: обучена
pre180 min_count=5 seed=5: обучена
pre180 min_count=5 seed=6: обучена
pre180 min_count=5 seed=7: обучена
pre180 min_count=5 seed=8: обучена
pre180 min_count=5 seed=9: обучена
pre180 min_count=5 seed=10: обучена
pre180 min_count=50 seed=1: обучена
pre180 min_count=50 seed=2: обучена
pre180 min_count=50 seed=3: обучена
pre180 min_count=50 seed=4: обучена
pre180 min_count=50 seed=5: обучена
pre180 min_count=50 seed=6: обучена
pre180 min_count=50 seed=7: обучена
pre180 min_count=50 seed=8: обучена
pre180 min_count=50 seed=9: обучена
pre180 min_count=50 seed=10: обучена
post180 min_count=5 seed=1: обучена
post180 min_count=5 seed=2: обучена
post180 min_count=5 seed=3: обучена
post180 min_count=5 seed=4: обучена
post180 min_count=5 seed=5: обучена
post180 min_count=5 seed=6: обучена
post180 min_count=5 seed=7: обучена
post180 min_count=5 seed=8: обучена


**Далее:** `02_select_models.ipynb`. Модели обучены независимо; представители серий ещё не выбраны.